In [2]:
import numpy as np
import plotly.graph_objects as go
import pandas as pd
import utils
import ast
import re
from datetime import datetime

# Data Loading

In [2]:
spike_df = pd.read_csv('../../Data_processed/energy_df_spike_type.csv')
spike_df['tstp'] = pd.to_datetime(spike_df['tstp'])
spike_df['date'] = spike_df['tstp'].dt.date
spike_df['time'] = spike_df['tstp'].dt.time
spike_df = spike_df.drop(columns=['tstp'])
print(spike_df.head())

       LCLid  spike_type  spike_magnitude        date      time
0  MAC000002           0              1.0  2012-10-12  00:30:00
1  MAC000002           0              1.0  2012-10-12  01:00:00
2  MAC000002           0              1.0  2012-10-12  01:30:00
3  MAC000002           0              1.0  2012-10-12  02:00:00
4  MAC000002           0              1.0  2012-10-12  02:30:00


In [3]:
weather_df = utils.get_weather_df()
weather_df = utils.process_weather_df(weather_df)
# Get the daily weather data using average of the hourly data
weather_df['date'] = pd.to_datetime(weather_df['tstp']).dt.date
daily_weather = weather_df.groupby('date').mean()
daily_weather = daily_weather.reset_index()
daily_weather = daily_weather.drop(columns=['tstp'])
print(daily_weather.head())

         date  temperature  windSpeed  humidity
0  2011-11-01     0.486447   0.195410  0.789097
1  2011-11-02     0.474220   0.297096  0.835635
2  2011-11-03     0.550331   0.275370  0.853092
3  2011-11-04     0.501639   0.193465  0.884746
4  2011-11-05     0.469117   0.197625  0.897057


In [4]:
statistics_df = pd.read_csv('../../Data_processed/Statistics.csv')
print(statistics_df.head())

       LCLid      mean  median       std    min    max  gradient
0  MAC000002  0.252511   0.158  0.247080  0.000  2.994  0.109262
1  MAC000003  0.397263   0.166  0.614692  0.007  3.921  0.176724
2  MAC000005  0.095425   0.041  0.122226  0.010  1.979  0.057801
3  MAC000007  0.197888   0.115  0.234394  0.015  3.784  0.105829
4  MAC000008  0.363184   0.296  0.241725  0.000  3.581  0.100872


In [5]:
spike_df = spike_df.merge(statistics_df, on='LCLid', how='left')
print(spike_df.head())

       LCLid  spike_type  spike_magnitude        date      time      mean  \
0  MAC000002           0              1.0  2012-10-12  00:30:00  0.252511   
1  MAC000002           0              1.0  2012-10-12  01:00:00  0.252511   
2  MAC000002           0              1.0  2012-10-12  01:30:00  0.252511   
3  MAC000002           0              1.0  2012-10-12  02:00:00  0.252511   
4  MAC000002           0              1.0  2012-10-12  02:30:00  0.252511   

   median      std  min    max  gradient  
0   0.158  0.24708  0.0  2.994  0.109262  
1   0.158  0.24708  0.0  2.994  0.109262  
2   0.158  0.24708  0.0  2.994  0.109262  
3   0.158  0.24708  0.0  2.994  0.109262  
4   0.158  0.24708  0.0  2.994  0.109262  


In [6]:
# Check for missing values
missing_values = spike_df.isnull().sum()
print(missing_values)

LCLid                   0
spike_type              0
spike_magnitude         0
date                    0
time                    0
mean               296886
median             296886
std                296886
min                296886
max                296886
gradient           296886
dtype: int64


In [7]:
# removes the ids that are in the statistics file but not in the energy file
spike_df = spike_df.dropna()

In [8]:
# Check for missing values
missing_values = spike_df.isnull().sum()
print(missing_values)

LCLid              0
spike_type         0
spike_magnitude    0
date               0
time               0
mean               0
median             0
std                0
min                0
max                0
gradient           0
dtype: int64


# Dataframe Making

In [9]:
ids = spike_df['LCLid'].unique()    
test_id = np.random.choice(ids, 10)

test_df = spike_df[spike_df['LCLid'].isin(test_id)]

In [10]:
def extract_spike_info(group):
    spike_num = 0
    spike_date = []
    spike_times = []
    spike_durations = []
    spike_magnitudes = []
    spike_type = []

    i = 0
    while i < len(group):

        row = group.iloc[i]

        if spike_date == []:
            date = row['date'].strftime('%Y-%m-%d')
            spike_date.append(date)

        if row['spike_type'] == 1:
            j = 0
            magnitude = 0

            time = row['time'].strftime('%H:%M:%S')
            spike_times.append(time)

            while i + j < len(group) and group.iloc[i + j]['spike_type'] != 0:

                next_row = group.iloc[i + j]

                if next_row['spike_type'] in [2, 3]:
                    magnitude += next_row['spike_magnitude']
                    spike_type.append(next_row['spike_type'])

                j += 1

            spike_durations.append(j)
            spike_magnitudes.append(magnitude)
            spike_num += 1
            i += j
            
        else:
            i += 1

    # Only add these if they exist and have correct indexing
    date = group['date'].iloc[0]

    weather_row = daily_weather.loc[daily_weather['date'] == date]
    weather = weather_row.drop(columns=['date']).to_numpy().squeeze().tolist()

    statistics = group.drop(columns=['LCLid', 'date', 'time', 'spike_type', 'spike_magnitude']).iloc[0].to_numpy().tolist()
    return (spike_num, spike_date, spike_times, spike_type, spike_durations, spike_magnitudes, statistics, weather)

# Group by LCLid and date
tensors = spike_df.groupby(['LCLid', 'date']).apply(extract_spike_info)
tensors = tensors.drop(columns=['LCLid', 'date'])




ValueError: 7 columns passed, passed data had 8 columns

In [12]:
# Convert the resulting Series to a DataFrame
tensor_df = pd.DataFrame(tensors.tolist(), columns=['spike_num', 'spike_date', 'spike_times', 'spike_type', 'spike_durations', 'spike_magnitudes', 'ID-statistics', 'weather'])

print(tensor_df.head())

   spike_num    spike_date                               spike_times  \
0          3  [2012-10-12]            [11:30:00, 16:30:00, 18:30:00]   
1          2  [2012-10-13]                      [09:30:00, 17:30:00]   
2          2  [2012-10-14]                      [09:30:00, 18:00:00]   
3          4  [2012-10-15]  [08:00:00, 11:00:00, 16:30:00, 18:30:00]   
4          3  [2012-10-16]            [08:30:00, 12:00:00, 17:30:00]   

     spike_type spike_durations              spike_magnitudes  \
0     [2, 2, 3]       [3, 3, 4]         [0.663, 0.493, 0.886]   
1     [3, 2, 3]          [4, 5]                [0.933, 1.306]   
2     [2, 2, 3]          [6, 9]                [1.191, 1.085]   
3  [3, 2, 2, 3]    [3, 3, 3, 3]  [1.075, 0.551, 0.695, 1.164]   
4  [2, 3, 2, 3]       [3, 3, 5]         [0.408, 0.991, 1.413]   

                                       ID-statistics  \
0  [0.2525107493475829, 0.158, 0.2470795043314368...   
1  [0.2525107493475829, 0.158, 0.2470795043314368...   
2  [0.25

In [13]:
tensor_df.to_csv('../../Data_processed/spike_tensors.csv', index=False)

# 111

In [3]:
tensor_df = pd.read_csv('../../Data_processed/spike_tensors.csv')
print(tensor_df.head())

   spike_num      spike_date  \
0          3  ['2012-10-12']   
1          2  ['2012-10-13']   
2          2  ['2012-10-14']   
3          4  ['2012-10-15']   
4          3  ['2012-10-16']   

                                        spike_times    spike_type  \
0              ['11:30:00', '16:30:00', '18:30:00']     [2, 2, 3]   
1                          ['09:30:00', '17:30:00']     [3, 2, 3]   
2                          ['09:30:00', '18:00:00']     [2, 2, 3]   
3  ['08:00:00', '11:00:00', '16:30:00', '18:30:00']  [3, 2, 2, 3]   
4              ['08:30:00', '12:00:00', '17:30:00']  [2, 3, 2, 3]   

  spike_durations              spike_magnitudes  \
0       [3, 3, 4]         [0.663, 0.493, 0.886]   
1          [4, 5]                [0.933, 1.306]   
2          [6, 9]                [1.191, 1.085]   
3    [3, 3, 3, 3]  [1.075, 0.551, 0.695, 1.164]   
4       [3, 3, 5]         [0.408, 0.991, 1.413]   

                                       ID-statistics  \
0  [0.2525107493475829, 0.158

In [4]:
tensor_df['spike_times'] = tensor_df['spike_times'].apply(lambda x: eval(x))

# Define a function to convert time to an integer
def time_to_interval(time_str):
    time = datetime.strptime(time_str, '%H:%M:%S')
    # Calculate the interval number based on 30-minute intervals
    interval = (time.hour * 2) + (time.minute // 30) + 1
    return interval

# Process each row and convert spike times
def process_row(row):
    spike_times = row['spike_times']
    interval_values = [time_to_interval(time) for time in spike_times]
    return interval_values

# Apply the function to each row
tensor_df['spike_times_intervals'] = tensor_df.apply(process_row, axis=1)

# The 'spike_times_intervals' column now contains the integer intervals for each spike time
print(tensor_df.head())

   spike_num      spike_date                               spike_times  \
0          3  ['2012-10-12']            [11:30:00, 16:30:00, 18:30:00]   
1          2  ['2012-10-13']                      [09:30:00, 17:30:00]   
2          2  ['2012-10-14']                      [09:30:00, 18:00:00]   
3          4  ['2012-10-15']  [08:00:00, 11:00:00, 16:30:00, 18:30:00]   
4          3  ['2012-10-16']            [08:30:00, 12:00:00, 17:30:00]   

     spike_type spike_durations              spike_magnitudes  \
0     [2, 2, 3]       [3, 3, 4]         [0.663, 0.493, 0.886]   
1     [3, 2, 3]          [4, 5]                [0.933, 1.306]   
2     [2, 2, 3]          [6, 9]                [1.191, 1.085]   
3  [3, 2, 2, 3]    [3, 3, 3, 3]  [1.075, 0.551, 0.695, 1.164]   
4  [2, 3, 2, 3]       [3, 3, 5]         [0.408, 0.991, 1.413]   

                                       ID-statistics  \
0  [0.2525107493475829, 0.158, 0.2470795043314368...   
1  [0.2525107493475829, 0.158, 0.2470795043314368...

In [5]:
tensor_df = tensor_df.drop(columns=['spike_times'])
print(tensor_df.head())

   spike_num      spike_date    spike_type spike_durations  \
0          3  ['2012-10-12']     [2, 2, 3]       [3, 3, 4]   
1          2  ['2012-10-13']     [3, 2, 3]          [4, 5]   
2          2  ['2012-10-14']     [2, 2, 3]          [6, 9]   
3          4  ['2012-10-15']  [3, 2, 2, 3]    [3, 3, 3, 3]   
4          3  ['2012-10-16']  [2, 3, 2, 3]       [3, 3, 5]   

               spike_magnitudes  \
0         [0.663, 0.493, 0.886]   
1                [0.933, 1.306]   
2                [1.191, 1.085]   
3  [1.075, 0.551, 0.695, 1.164]   
4         [0.408, 0.991, 1.413]   

                                       ID-statistics  \
0  [0.2525107493475829, 0.158, 0.2470795043314368...   
1  [0.2525107493475829, 0.158, 0.2470795043314368...   
2  [0.2525107493475829, 0.158, 0.2470795043314368...   
3  [0.2525107493475829, 0.158, 0.2470795043314368...   
4  [0.2525107493475829, 0.158, 0.2470795043314368...   

                                             weather spike_times_intervals  
0 

In [6]:
def convert_date_to_sin_cos(date):
    # Ensure date is a string, then process
    if isinstance(date, str):
        date_str = eval(date)[0]
    else:
        date_str = date[0]  # If date is a list or tuple, access first element directly
    
    day_of_year = pd.to_datetime(date_str).dayofyear
    date_sin = np.sin(day_of_year * (2 * np.pi / 365))
    date_cos = np.cos(day_of_year * (2 * np.pi / 365))
    return date_sin, date_cos

tensor_df['date_sin'], tensor_df['date_cos'] = zip(*tensor_df['spike_date'].apply(convert_date_to_sin_cos))
print(tensor_df.head())


   spike_num      spike_date    spike_type spike_durations  \
0          3  ['2012-10-12']     [2, 2, 3]       [3, 3, 4]   
1          2  ['2012-10-13']     [3, 2, 3]          [4, 5]   
2          2  ['2012-10-14']     [2, 2, 3]          [6, 9]   
3          4  ['2012-10-15']  [3, 2, 2, 3]    [3, 3, 3, 3]   
4          3  ['2012-10-16']  [2, 3, 2, 3]       [3, 3, 5]   

               spike_magnitudes  \
0         [0.663, 0.493, 0.886]   
1                [0.933, 1.306]   
2                [1.191, 1.085]   
3  [1.075, 0.551, 0.695, 1.164]   
4         [0.408, 0.991, 1.413]   

                                       ID-statistics  \
0  [0.2525107493475829, 0.158, 0.2470795043314368...   
1  [0.2525107493475829, 0.158, 0.2470795043314368...   
2  [0.2525107493475829, 0.158, 0.2470795043314368...   
3  [0.2525107493475829, 0.158, 0.2470795043314368...   
4  [0.2525107493475829, 0.158, 0.2470795043314368...   

                                             weather spike_times_intervals  \
0

In [7]:
tensor_df = tensor_df.drop(columns=['spike_date'])
print(tensor_df.head())

   spike_num    spike_type spike_durations              spike_magnitudes  \
0          3     [2, 2, 3]       [3, 3, 4]         [0.663, 0.493, 0.886]   
1          2     [3, 2, 3]          [4, 5]                [0.933, 1.306]   
2          2     [2, 2, 3]          [6, 9]                [1.191, 1.085]   
3          4  [3, 2, 2, 3]    [3, 3, 3, 3]  [1.075, 0.551, 0.695, 1.164]   
4          3  [2, 3, 2, 3]       [3, 3, 5]         [0.408, 0.991, 1.413]   

                                       ID-statistics  \
0  [0.2525107493475829, 0.158, 0.2470795043314368...   
1  [0.2525107493475829, 0.158, 0.2470795043314368...   
2  [0.2525107493475829, 0.158, 0.2470795043314368...   
3  [0.2525107493475829, 0.158, 0.2470795043314368...   
4  [0.2525107493475829, 0.158, 0.2470795043314368...   

                                             weather spike_times_intervals  \
0  [0.44636145833333335, 0.3711697916666667, 0.66...          [24, 34, 38]   
1  [0.37326145833333335, 0.1712510416666667, 0.80.

In [8]:
tensor_df.to_csv('../../Data_processed/spike_tensors_x.csv', index=False)

# sin_cos

In [9]:
# def convert_time_to_sin_cos(times):
#     time_sin_cos = []
#     for time in times:
#         h, m, s = map(int, time.split(':'))
#         total_minutes = h * 60 + m
#         time_sin = np.sin(total_minutes * (2 * np.pi / 1440))
#         time_cos = np.cos(total_minutes * (2 * np.pi / 1440))
#         time_sin_cos.append([time_sin, time_cos])
#     return time_sin_cos

# tensor_df['time_sin_cos'] = tensor_df['spike_times'].apply(convert_time_to_sin_cos)

# print(tensor_df.head())

In [10]:
# tensor_df = tensor_df.drop(columns=['spike_date', 'spike_times'])
# print(tensor_df.head())

In [11]:
# tensor_df.to_csv('../../Data_processed/spike_tensors.csv', index=False)